In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger
from keras.callbacks import EarlyStopping

import os
import random
from datetime import datetime
import time

import matplotlib.pyplot as plt

pi = 3.14159265359
maxval=1e9
minval=1e-9

2025-07-22 15:30:43.872431: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-07-22 15:30:43.872506: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-07-22 15:30:43.873406: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-22 15:30:43.879817: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-22 15:30:45.026346: W tensorflow/compiler/tf2

In [2]:
# # os.chdir('SmartPix/data_generator')
os.chdir('/home/das214/SmartPix/SoftQuantize')
!pwd

/home/das214/SmartPix/SoftQuantize


In [3]:
from DG.OptimizedDataGenerator_v2 import OptimizedDataGenerator
from losses.loss import custom_loss
from models.SoftQuantizeLayer import SoftQuantizeLayer
from models.AnnealingScheduler import AnnealingScheduler
# from models.models import CreateModel # Conv2D model

In [4]:
import keras
from keras.layers import *
from keras.models import Sequential, Model
from keras.utils import Sequence
from qkeras import *

import tensorflow as tf
from tensorflow.keras import datasets, layers, models

def var_network(var, hidden=10, output=2):
    var = Flatten()(var)
    var = QDense(
        hidden,
        kernel_quantizer=quantized_bits(8, 0, alpha=1),
        bias_quantizer=quantized_bits(8, 0, alpha=1),
        kernel_regularizer=tf.keras.regularizers.L1L2(0.01),
        activity_regularizer=tf.keras.regularizers.L2(0.01),
    )(var)
    var = QActivation("quantized_tanh(8, 0, 1)")(var)
    var = QDense(
        hidden,
        kernel_quantizer=quantized_bits(8, 0, alpha=1),
        bias_quantizer=quantized_bits(8, 0, alpha=1),
        kernel_regularizer=tf.keras.regularizers.L1L2(0.01),
        activity_regularizer=tf.keras.regularizers.L2(0.01),
    )(var)
    var = QActivation("quantized_tanh(8, 0, 1)")(var)
    return QDense(
        output,
        kernel_quantizer=quantized_bits(8, 0, alpha=1),
        bias_quantizer=quantized_bits(8, 0, alpha=1),
        kernel_regularizer=tf.keras.regularizers.L1L2(0.01),
    )(var)

def conv_network(var, n_filters=5, kernel_size=3):
    var = QSeparableConv2D(
        n_filters,kernel_size,
        depthwise_quantizer=quantized_bits(4, 0, 1, alpha=1),
        pointwise_quantizer=quantized_bits(4, 0, 1, alpha=1),
        bias_quantizer=quantized_bits(4, 0, alpha=1),
        depthwise_regularizer=tf.keras.regularizers.L1L2(0.01),
        pointwise_regularizer=tf.keras.regularizers.L1L2(0.01),
        activity_regularizer=tf.keras.regularizers.L2(0.01),
    )(var)
    var = QActivation("quantized_tanh(4, 0, 1)")(var)
    var = QConv2D(
        n_filters,1,
        kernel_quantizer=quantized_bits(4, 0, alpha=1),
        bias_quantizer=quantized_bits(4, 0, alpha=1),
        kernel_regularizer=tf.keras.regularizers.L1L2(0.01),
        activity_regularizer=tf.keras.regularizers.L2(0.01),
    )(var)
    var = QActivation("quantized_tanh(4, 0, 1)")(var)    
    return var

def CreateModel(shape, output, n_filters, pool_size):
    x_base = x_in = Input(shape)
    x_base = SoftQuantizeLayer(
        n_bits=2,                     
        initial_range=[-1.0, 1.0],    
        trainable_levels=False,        
        trainable_bins=True,          
        initial_k=1.0,                
        trainable_k=True,             
        name='soft_quantizer_output'  
    )(x_base)

    stack = conv_network(x_base)
    stack = AveragePooling2D(
        pool_size=(pool_size, pool_size), 
        strides=None, 
        padding="valid", 
        data_format=None,        
    )(stack)
    stack = QActivation("quantized_bits(8, 0, alpha=1)")(stack)
    stack = var_network(stack, hidden=16, output=output)
    model = Model(inputs=x_in, outputs=stack)
    return model

In [5]:
dataset_base_dir = "/depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained"
tfrecords_base_dir = os.path.join(dataset_base_dir, "TFR_files", "2t")

dataset_train_dir = os.path.join(dataset_base_dir, "train")
dataset_test_dir = os.path.join(dataset_base_dir, "test")
tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train")
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val")

batch_size = 5000
val_batch_size = 5000
train_file_size = len(os.listdir(dataset_train_dir))
val_file_size = len(os.listdir(dataset_test_dir))

In [6]:
# start_time = time.time()
# validation_generator = OptimizedDataGenerator(
#     dataset_base_dir = dataset_test_dir,
#     file_type = "parquet",
#     data_format = "3D",
#     batch_size = val_batch_size,
#     # optimize_batch_size = True,
#     file_count = val_file_size,
#     to_standardize= True,
#     labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
#     input_shape = (2,16,16), # (20,13,21),
#     transpose = (0,2,3,1),
#     shuffle = False, 
#     files_from_end=True,

#     tfrecords_dir = tfrecords_dir_val,
#     use_time_stamps = [0,19],
#     max_workers = 2
# )

# print("--- Validation generator %s seconds ---" % (time.time() - start_time))

# # training generator
# start_time = time.time()
# training_generator = OptimizedDataGenerator(
#     dataset_base_dir = dataset_train_dir,
#     file_type = "parquet",
#     data_format = "3D",
#     batch_size = batch_size,
#     # optimize_batch_size = True,
#     file_count = train_file_size,
#     to_standardize= True,
#     labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
#     input_shape = (2,16,16), # (20,13,21),
#     transpose = (0,2,3,1),
#     shuffle = False, # True 

#     tfrecords_dir = tfrecords_dir_train,
#     use_time_stamps = [0,19],
#     max_workers = 2
# )
# print("--- Training generator %s seconds ---" % (time.time() - start_time))

In [7]:
# Loading pre-generated TFRecords
validation_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir= tfrecords_dir_val,
    shuffle=True,
    seed=42,
    quantize=False,
)

training_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_train,
    shuffle=True,
    seed=42,
    quantize=False,
)


Loading metadata from /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_val/metadata.json
Loading metadata from /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_train/metadata.json


In [8]:
model=CreateModel(shape = (16,16,2), output = 14, n_filters=5,pool_size=3)
model.compile(
    optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3, clipnorm=1.0),
    loss=custom_loss,
)

model.summary()

2025-07-22 15:30:49.900615: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 37496 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB MIG 7g.40gb, pci bus id: 0000:21:00.0, compute capability: 8.0
2025-07-22 15:30:50.260571: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 16, 16, 2)]       0         
                                                                 
 soft_quantizer_output (Sof  (None, 16, 16, 2)         9         
 tQuantizeLayer)                                                 
                                                                 
 q_separable_conv2d (QSepar  (None, 14, 14, 5)         33        
 ableConv2D)                                                     
                                                                 
 q_activation (QActivation)  (None, 14, 14, 5)         0         
                                                                 
 q_conv2d (QConv2D)          (None, 14, 14, 5)         30        
                                                                 
 q_activation_1 (QActivatio  (None, 14, 14, 5)         0     

In [9]:
from datetime import datetime

fingerprint = '%08x' % random.randrange(16**8)
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
os.makedirs("trained_models", exist_ok=True)
base_dir = f'./trained_models/model-{fingerprint}-checkpoints'

checkpoints_dir = os.path.join(base_dir, 'checkpoints')

os.makedirs(base_dir, exist_ok=True)
os.makedirs(checkpoints_dir, exist_ok=True) 
checkpoint_filepath = os.path.join(checkpoints_dir, 'weights.{epoch:02d}-t{loss:.2f}-v{val_loss:.2f}.hdf5')

In [10]:
checkpoint_filepath

'./trained_models/model-41166c06-checkpoints/checkpoints/weights.{epoch:02d}-t{loss:.2f}-v{val_loss:.2f}.hdf5'

In [11]:
# 9f5e5c2c : 1000 epochs
print(fingerprint)

41166c06


In [12]:
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback
import csv

early_stopping_patience = 50
es = EarlyStopping(patience=early_stopping_patience, restore_best_weights=True)

mcp = ModelCheckpoint(
        filepath=checkpoint_filepath,
        save_weights_only=True,
       save_freq='epoch'
)

class LevelsLoggerCallback(Callback):
    """
    Logs k, quantization levels, and bin centers to a CSV file each epoch.
    """
    def __init__(self, log_filepath, layer_name="soft_quantizer_output"):
        super().__init__()
        self.log_filepath = log_filepath
        self.layer_name = layer_name

    def on_train_begin(self, logs=None):
        os.makedirs(os.path.dirname(self.log_filepath), exist_ok=True)
        try:
            layer = self.model.get_layer(self.layer_name)
            num_levels = layer.num_levels
        except AttributeError:
            print(f"Warning: Layer '{self.layer_name}' might not be a SoftQuantizeLayer.")
            num_levels = 4 
        
        level_headers = [f'level_{i}' for i in range(num_levels)]
        bin_headers = [f'bin_center_{i}' for i in range(num_levels)]
        self.header = ['epoch', 'k'] + level_headers + bin_headers

        if not os.path.exists(self.log_filepath):
            with open(self.log_filepath, mode='w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(self.header)

    def on_epoch_end(self, epoch, logs=None):
        """Appends the latest k, levels, and bins to the CSV file."""
        layer = self.model.get_layer(self.layer_name)
        
        k_val = layer.k.numpy()[0]
        levels = layer.levels.numpy().tolist()
        bins = layer.bin_centers.numpy().tolist()
        
        row_data = [epoch, k_val] + levels + bins
        
        with open(self.log_filepath, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(row_data)

csv_logger = CSVLogger(f'{base_dir}/training_log.csv', append=True)
scheduler_callback = AnnealingScheduler(
    schedule='cosine',  
    target_layer_name='soft_quantizer_output', 
    initial_k=1.0,
    final_k=67.0, 
    verbose=1      
)
levels_logger = LevelsLoggerCallback(
    log_filepath=f"{base_dir}/quant_levels_log.csv",
    layer_name="soft_quantizer_output"
)

In [13]:
history = model.fit(
        x=training_generator,
        validation_data=validation_generator,
        callbacks=[mcp, csv_logger, scheduler_callback, levels_logger],
        epochs=1000,
        shuffle=False,
        verbose=1
    )


Epoch 1: Annealing 'k' set to 1.0000
	Levels: -1.0000, -0.3333, 0.3333, 1.0000
Epoch 1/1000


2025-07-22 15:30:56.545434: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
2025-07-22 15:30:56.629992: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2025-07-22 15:30:56.921661: I tensorflow/core/util/cuda_solvers.cc:179] Creating GpuSolver handles for stream 0x55efb8b06b00
2025-07-22 15:30:59.348014: I external/local_xla/xla/service/service.cc:168] XLA service 0x7fa060a07770 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-07-22 15:30:59.348065: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100-PCIE-40GB MIG 7g.40gb, Compute Capability 8.0
2025-07-22 15:30:59.359434: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1753191059.469540 2797857 device_compiler.h:186] Compile

84/84 [==============================] - 19s 130ms/step - loss: 47219.0117 - val_loss: 16182.2559

Epoch 2: Annealing 'k' set to 1.0002
	Levels: -1.0000, -0.3333, 0.3333, 1.0000
Epoch 2/1000
84/84 [==============================] - 8s 89ms/step - loss: 13870.8652 - val_loss: 12290.0078

Epoch 3: Annealing 'k' set to 1.0007
	Levels: -1.0000, -0.3333, 0.3333, 1.0000
Epoch 3/1000
84/84 [==============================] - 7s 87ms/step - loss: 20665.3398 - val_loss: 12692.2900

Epoch 4: Annealing 'k' set to 1.0015
	Levels: -1.0000, -0.3333, 0.3333, 1.0000
Epoch 4/1000
84/84 [==============================] - 7s 86ms/step - loss: 17453.5645 - val_loss: 102853.2109

Epoch 5: Annealing 'k' set to 1.0026
	Levels: -1.0000, -0.3333, 0.3333, 1.0000
Epoch 5/1000
84/84 [==============================] - 7s 87ms/step - loss: 9954.7314 - val_loss: 7096.4434

Epoch 6: Annealing 'k' set to 1.0041
	Levels: -1.0000, -0.3333, 0.3333, 1.0000
Epoch 6/1000
84/84 [==============================] - 8s 90ms/step 

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



84/84 [==============================] - 7s 85ms/step - loss: -20492.3125 - val_loss: -21212.5137

Epoch 492: Annealing 'k' set to 33.0671
	Levels: -1.0000, -0.3333, 0.3333, 1.0000
Epoch 492/1000
84/84 [==============================] - 7s 85ms/step - loss: -20027.3672 - val_loss: -19371.3691

Epoch 493: Annealing 'k' set to 33.1707
	Levels: -1.0000, -0.3333, 0.3333, 1.0000
Epoch 493/1000
84/84 [==============================] - 7s 84ms/step - loss: -20395.6309 - val_loss: -17204.2832

Epoch 494: Annealing 'k' set to 33.2743
	Levels: -1.0000, -0.3333, 0.3333, 1.0000
Epoch 494/1000
84/84 [==============================] - 7s 84ms/step - loss: -20660.9297 - val_loss: -18638.8574

Epoch 495: Annealing 'k' set to 33.3780
	Levels: -1.0000, -0.3333, 0.3333, 1.0000
Epoch 495/1000
84/84 [==============================] - 7s 84ms/step - loss: -20295.4727 - val_loss: -19061.5938

Epoch 496: Annealing 'k' set to 33.4817
	Levels: -1.0000, -0.3333, 0.3333, 1.0000
Epoch 496/1000
84/84 [=============

In [14]:
1

1

In [15]:
sq_layer = model.get_layer(name="soft_quantizer_output")

# Access its parameters
print("Initial Levels:", sq_layer.levels.numpy())  # or sq_layer.levels if it's not a tf.Variable
print("Current k:", sq_layer.k.numpy())

Initial Levels: [-1.         -0.3333333   0.33333337  1.        ]
Current k: [70.99392]
